## Common Data Types encountered in LangGraph

In [1]:
%load_ext nb_mypy
%nb_mypy On
# %nb_mypy mypy-options --strict

Version 1.0.6


### Data Type - TypedDict

A TypedDict behaves like a regular dictionary at Runtime

Additionally, it allows defining 
* the names of the keys in the dictionary
* the data types of the values for each key

Code Editors and Static Type Checkers use this information to check if the dictionary is constructed as per the specifications

In [2]:
from typing import TypedDict, Optional, NotRequired

In [3]:
class AgentState(TypedDict):
    client_name: str
    client_age: NotRequired[int]
    request_type: str
    dollar_amount: Optional[float]

In [4]:
state : AgentState = {
    'client_name': 'John Doe',
    'client_age': 62,
    'request_type': 'Loan Application',
    'dollar_amount' : 12_000.00
}

In [5]:
state : AgentState = {
    'client_name': 'John Doe',
    # 'client_age': 62,  # missing
    'request_type': 'Loan Application',
    'dollar_amount' : 12_000.00
}

In [6]:
state : AgentState = {
    'client_name': 'John Doe',
    'client_age': '62',  # bad data type
    'request_type': 'Loan Application',
    'dollar_amount' : 12_000.00
}

<cell>3: error: Incompatible types (expression has type "str", TypedDict item "client_age" has type "int")  [typeddict-item]


In [7]:
state : AgentState = {
    'client_name': 'John Doe',
    'client_age': 62,
    'request_type': 'Loan Application',
    # 'dollar_amount' : 12_000.00 # missing
}

<cell>1: error: Missing key "dollar_amount" for TypedDict "AgentState"  [typeddict-item]


In [8]:
state : AgentState = {
    'client_name': 'John Doe',
    'client_age': 62,
    'request_type': 'Loan Application',
    'dollar_amount' : None # empty
}

In [9]:
state : AgentState = {
    'client_name': 'John Doe',
    'client_age': 62,
    'request_type': 'Loan Application',
    # 'dollar_amount' : 12_000 # missing
}

<cell>1: error: Missing key "dollar_amount" for TypedDict "AgentState"  [typeddict-item]


In [10]:
state : AgentState = {
    'client_name': 'John Doe',
    'client_age': 62,
    'request_type': 'Loan Application',
    'dollar_amount' : 12_000,
    'request_date': '2026-10-10' # unexpected
}

<cell>1: error: Extra key "request_date" for TypedDict "AgentState"  [typeddict-unknown-key]


Applications built with LangGraph define a centralized **State** object to track and pass information across a workflow, such as managing the stages of a loan application.

As workflows grow, these state definitions often scale in complexity to hold a wide variety of diverse attributes. To manage this safely, LangGraph recommends using `TypedDict`. 

This approach provides developers with real-time autocompletion and allows static type checkers (like Mypy) to catch bugs early, flag typos, or highlight missing keys before the code ever runs.

## Data Type - BaseModel (from Pydantic)

In [11]:
from pydantic import BaseModel
from typing import Literal

Pydantic allows definition of a scheme for data objects

Makes best effort to parse real world data to match scheme

In [12]:
TransactionCategory = Literal[
    'INCOME_SALARY_RCVD',
    'GROCERIES_AND_ESSENTIALS_PAID',
    'TRAVEL_AND_TRANSPORT_PAID',
]

class Transaction(BaseModel):
    date: str
    amount: float
    description: str
    category: Optional[TransactionCategory] = None

In [13]:
transaction_data = {
    'date': '2026-10-01',
    'amount': 35172.00,
    'description': 'ABC Corp. Pay Day',
    'category': 'INCOME_SALARY_RCVD'
}

tx = Transaction.model_validate(transaction_data)
tx

Transaction(date='2026-10-01', amount=35172.0, description='ABC Corp. Pay Day', category='INCOME_SALARY_RCVD')

In [14]:
transaction_data = {
    'date': '2026-10-01',
    'amount': '35172.00', # numeric value represented as string
    'description': 'ABC Corp. Pay Day',
    'category': 'INCOME_SALARY_RCVD'
}

tx = Transaction.model_validate(transaction_data)
tx

Transaction(date='2026-10-01', amount=35172.0, description='ABC Corp. Pay Day', category='INCOME_SALARY_RCVD')

In [15]:
transaction_data = {
    'date': '2026-10-01',
    'amount': 35172.00,
    'description': 'ABC Corp. Pay Day',
    'category': None # empty
}

tx = Transaction.model_validate(transaction_data)
tx

Transaction(date='2026-10-01', amount=35172.0, description='ABC Corp. Pay Day', category=None)

In [16]:
transaction_data = {
    'date': '2026-10-01',
    'amount': 35172.00,
    'description': 'ABC Corp. Pay Day',
    # 'category': 'INCOME_SALARY_RCVD' # missing
}

tx = Transaction.model_validate(transaction_data)
tx

Transaction(date='2026-10-01', amount=35172.0, description='ABC Corp. Pay Day', category=None)

In [17]:
transaction_data = {
    'date': '10 Oct 2026', # alternate date format
    'amount': 35172.00,
    'description': 'ABC Corp. Pay Day',
    'category': 'INCOME_SALARY_RCVD'
}

tx = Transaction.model_validate(transaction_data)
tx

Transaction(date='10 Oct 2026', amount=35172.0, description='ABC Corp. Pay Day', category='INCOME_SALARY_RCVD')

In [18]:
transaction_data = {
    'date': '2026-10-01',
    'amount': '$35172.00', # leading $
    'description': 'ABC Corp. Pay Day',
    'category': 'INCOME_SALARY_RCVD'
}

try:
    tx = Transaction.model_validate(transaction_data)
except Exception as e:
    print(e)

1 validation error for Transaction
amount
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='$35172.00', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/float_parsing


In [19]:
transaction_data = {
    'date': '2026-10-01',
    'amount': 35172.00,
    'description': 'ABC Corp. Pay Day',
    'category': 'MISC_PAYMENT' # Bad value
}

try:
    tx = Transaction.model_validate(transaction_data)
except Exception as e:
    print(e)

1 validation error for Transaction
category
  Input should be 'INCOME_SALARY_RCVD', 'GROCERIES_AND_ESSENTIALS_PAID' or 'TRAVEL_AND_TRANSPORT_PAID' [type=literal_error, input_value='MISC_PAYMENT', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error


### Print json schema that object should confirm to

In [20]:
Transaction.model_json_schema()

{'properties': {'date': {'title': 'Date', 'type': 'string'},
  'amount': {'title': 'Amount', 'type': 'number'},
  'description': {'title': 'Description', 'type': 'string'},
  'category': {'anyOf': [{'enum': ['INCOME_SALARY_RCVD',
      'GROCERIES_AND_ESSENTIALS_PAID',
      'TRAVEL_AND_TRANSPORT_PAID'],
     'type': 'string'},
    {'type': 'null'}],
   'default': None,
   'title': 'Category'}},
 'required': ['date', 'amount', 'description'],
 'title': 'Transaction',
 'type': 'object'}

## Data Type - Field (from Pydantic)

### Enhance schema with field descriptions

`Field` allows enhancing the schema with a custom description

In [21]:
from pydantic import Field

In [22]:
class Transaction(BaseModel):
    date: str = Field(description='Date in YYYY-MM-DD format')
    amount: float = Field(description='Exact transaction amount as is from Bank Statement')
    description: str = Field(description='Description of Transaction as is from Bank Statement')
    category: TransactionCategory = Field(description='Broad category assigned to the given transaction')

In [23]:
Transaction.model_json_schema()

{'properties': {'date': {'description': 'Date in YYYY-MM-DD format',
   'title': 'Date',
   'type': 'string'},
  'amount': {'description': 'Exact transaction amount as is from Bank Statement',
   'title': 'Amount',
   'type': 'number'},
  'description': {'description': 'Description of Transaction as is from Bank Statement',
   'title': 'Description',
   'type': 'string'},
  'category': {'description': 'Broad category assigned to the given transaction',
   'enum': ['INCOME_SALARY_RCVD',
    'GROCERIES_AND_ESSENTIALS_PAID',
    'TRAVEL_AND_TRANSPORT_PAID'],
   'title': 'Category',
   'type': 'string'}},
 'required': ['date', 'amount', 'description', 'category'],
 'title': 'Transaction',
 'type': 'object'}

### Enhance schema with field validation

`Field` allows to add custom validation checks (e.g. min_length, max_length, gt, ge, lt, le)

In [24]:
class Transaction(BaseModel):
    date: str = Field(description='Date in YYYY-MM-DD format')
    amount: float = Field(description='Exact transaction amount as is from Bank Statement')
    description: str = Field(description='Description of Transaction as is from Bank Statement', min_length=3)
    category: TransactionCategory = Field(description='Broad category assigned to the given transaction')

In [25]:
Transaction.model_json_schema()

{'properties': {'date': {'description': 'Date in YYYY-MM-DD format',
   'title': 'Date',
   'type': 'string'},
  'amount': {'description': 'Exact transaction amount as is from Bank Statement',
   'title': 'Amount',
   'type': 'number'},
  'description': {'description': 'Description of Transaction as is from Bank Statement',
   'minLength': 3,
   'title': 'Description',
   'type': 'string'},
  'category': {'description': 'Broad category assigned to the given transaction',
   'enum': ['INCOME_SALARY_RCVD',
    'GROCERIES_AND_ESSENTIALS_PAID',
    'TRAVEL_AND_TRANSPORT_PAID'],
   'title': 'Category',
   'type': 'string'}},
 'required': ['date', 'amount', 'description', 'category'],
 'title': 'Transaction',
 'type': 'object'}

In [26]:
transaction_data = {
    'date': '2026-10-01',
    'amount': 35172.00,
    'description': 'ABC Corp. Pay Day',
    'category': 'INCOME_SALARY_RCVD'
}

tx = Transaction.model_validate(transaction_data)

In [27]:
transaction_data = {
    'date': '2026-10-01',
    'amount': 35172.00,
    'description': 'PD', # length = 2
    'category': 'INCOME_SALARY_RCVD'
}

try:
    tx = Transaction.model_validate(transaction_data)
except Exception as e:
    print(e)

1 validation error for Transaction
description
  String should have at least 3 characters [type=string_too_short, input_value='PD', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/string_too_short
